# Skin Cancer Detection — Evaluation & Experiments
**Group Project | HAM10000 Dataset | Two-Stage Hierarchical Pipeline**

**Author:** Caden

---

This notebook covers the full Experiments section of the report:
1. Experimental Setup
2. Load Real Predictions
3. Quantitative Metrics
4. Pipeline Evaluation
5. Model Comparison
6. Training Curves
7. Grad-CAM / Attention Rollout Visualisation
8. Efficiency Comparison
9. Limitations & Conclusion

---
## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, sys
from pathlib import Path
from PIL import Image

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    recall_score, precision_score, f1_score, accuracy_score,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

sys.path.insert(0, str(Path('.').resolve()))
from evaluation import (
    evaluate_stage1, evaluate_stage2, evaluate_pipeline,
    compare_models, BINARY_CLASSES, MALIGNANT_CLASSES
)

PRED_DIR = Path('predictions')
FIG_DIR  = Path('figures_pretrained')
print('All imports successful.')

---
## 2. Experimental Setup

### Data Split

The HAM10000 dataset (10,015 images) was split **70/15/15** (stratified by lesion ID to prevent
data leakage from duplicate dermoscopy images of the same lesion):

| Split | % | Images |
|---|---|---|
| Training | 70% | 7,010 |
| Validation | 15% | 1,502 |
| Test | 15% | **1,503** |

The same test indices (saved to `test_indices.csv`) were used for both models to ensure
results are directly comparable.

### Preprocessing & Augmentation

All images resized to **224×224** and normalised with ImageNet mean/std.
Training augmentation: random horizontal/vertical flips, rotation ±30°, colour jitter, random cropped resize.

### Training Configuration

| Setting | MobileNetV2 | ViT-B/16 |
|---|---|---|
| Backbone | ImageNet pretrained (frozen then unfrozen) | ImageNet pretrained (frozen then unfrozen) |
| Optimiser | Adam | AdamW |
| Stage 1 head LR | 1e-4 | 1e-3 |
| Stage 1 backbone LR | — (unfroze last 4 blocks) | 1e-5 |
| Stage 2 head LR | 1e-4 | 1e-4 |
| Stage 2 backbone LR | — (unfroze last 4 blocks) | 1e-5 |
| Batch size | 32 | 32 |
| Loss | CrossEntropyLoss + class weights | CrossEntropyLoss + class weights |
| Class imbalance | Inverse-frequency weights | Inverse-frequency weights |

---
## 3. Load Real Predictions

Predictions were generated by running both trained models on the shared 1,503-image test set
using `generate_predictions.py`. Arrays are loaded from `predictions/`.

In [ ]:
# Ground truth
y_true_binary = np.load(PRED_DIR / 'y_true_binary.npy')

# MobileNetV2
mn_pred_binary  = np.load(PRED_DIR / 'mn_pred_binary.npy')
mn_probs_binary = np.load(PRED_DIR / 'mn_probs_binary.npy')
mn_true_s2      = np.load(PRED_DIR / 'mn_true_s2.npy').astype(int)
mn_pred_s2      = np.load(PRED_DIR / 'mn_pred_s2.npy').astype(int)
mn_probs_s2     = np.load(PRED_DIR / 'mn_probs_s2.npy')

# ViT-B/16
vit_pred_binary  = np.load(PRED_DIR / 'vit_pred_binary.npy')
vit_probs_binary = np.load(PRED_DIR / 'vit_probs_binary.npy')
vit_true_s2      = np.load(PRED_DIR / 'vit_true_s2.npy').astype(int)
vit_pred_s2      = np.load(PRED_DIR / 'vit_pred_s2.npy').astype(int)
vit_probs_s2     = np.load(PRED_DIR / 'vit_probs_s2.npy')

print(f'Test set:   {len(y_true_binary)} images')
print(f'Malignant:  {(y_true_binary==1).sum()} ({(y_true_binary==1).mean()*100:.1f}%)')
print(f'Benign:     {(y_true_binary==0).sum()} ({(y_true_binary==0).mean()*100:.1f}%)')
print()
print(f'MobileNetV2 flagged malignant (Stage 1): {mn_pred_binary.sum()}')
print(f'ViT-B/16   flagged malignant (Stage 1): {vit_pred_binary.sum()}')

---
## 4. Quantitative Metrics

### Why each metric was chosen

| Metric | Importance | Reason |
|---|---|---|
| **Recall (Sensitivity)** | Highest | Measures how many cancer cases we catch. A low recall means missed cancers — the most dangerous error. |
| **F1-Score** | High | Balances precision and recall. More meaningful than accuracy on imbalanced data. |
| **ROC-AUC** | High | Measures discriminative ability across all thresholds, threshold-independent. |
| **Accuracy** | Low | Misleading here — HAM10000 is 80% benign, so a trivial model scores ~80%. |

### 4.1 MobileNetV2 — Stage 1

In [ ]:
mn_s1_results = evaluate_stage1(y_true_binary, mn_pred_binary, mn_probs_binary)

### 4.2 MobileNetV2 — Stage 2

In [ ]:
mn_s2_results = evaluate_stage2(mn_true_s2, mn_pred_s2, mn_probs_s2)

### 4.3 ViT-B/16 — Stage 1

In [ ]:
vit_s1_results = evaluate_stage1(y_true_binary, vit_pred_binary, vit_probs_binary)

### 4.4 ViT-B/16 — Stage 2

In [ ]:
vit_s2_results = evaluate_stage2(vit_true_s2, vit_pred_s2, vit_probs_s2)

---
## 5. Pipeline Evaluation

The pipeline measures end-to-end performance: a malignant case must be correctly flagged by
Stage 1 **and** correctly subtyped by Stage 2 to count as a full pipeline success.
Any case missed by Stage 1 is never seen by Stage 2 — this is called *leakage*.

### 5.1 MobileNetV2 — Full Pipeline

In [ ]:
mn_pipeline = evaluate_pipeline(y_true_binary, mn_pred_binary, mn_true_s2, mn_pred_s2)

### 5.2 ViT-B/16 — Full Pipeline

In [ ]:
vit_pipeline = evaluate_pipeline(y_true_binary, vit_pred_binary, vit_true_s2, vit_pred_s2)

---
## 6. Model Comparison

Side-by-side comparison of all key metrics for both models.

In [ ]:
mn_all  = {**mn_s1_results,  **mn_s2_results,  **mn_pipeline}
vit_all = {**vit_s1_results, **vit_s2_results, **vit_pipeline}

mn_num  = {k: v for k, v in mn_all.items()  if isinstance(v, (int, float))}
vit_num = {k: v for k, v in vit_all.items() if isinstance(v, (int, float))}

comparison_df = compare_models(mn_num, vit_num)

---
## 7. Training Curves

Validation weighted F1 per epoch for both models during Stage 1 and Stage 2 training.
Both models use a two-phase approach: head-only warm-up followed by backbone fine-tuning.

In [ ]:
# ── MobileNetV2 Stage 1 ──────────────────────────────────────────────────────
# Warm-up (5 epochs, head only) + Fine-tune (5 epochs, last 4 blocks unfrozen)
# Source: skin_cancer_pipeline-checkpoint.ipynb training output
# Metric tracked: malignant recall (primary optimisation target)
mn_s1_warmup_recall = [0.9966, 0.9829, 0.9727, 0.9659, 0.9693]
mn_s1_finetune_recall= [0.9659, 0.9556, 0.9556, 0.9590, 0.9488]
mn_s1_val_recall = mn_s1_warmup_recall + mn_s1_finetune_recall

# ── ViT-B/16 Stage 1 ─────────────────────────────────────────────────────────
# Phase A (5 epochs, head only) + Phase B (10 epochs, full backbone)
# Source: vit_pretrained_ham10000.ipynb training output
# Metric tracked: validation weighted F1
vit_s1_phaseA_f1 = [0.7611, 0.7882, 0.8101, 0.8271, 0.8082]
vit_s1_phaseB_f1 = [0.8326, 0.8434, 0.8090, 0.7935, 0.8622,
                    0.8233, 0.8340, 0.8371, 0.8572, 0.8346]
vit_s1_val_f1 = vit_s1_phaseA_f1 + vit_s1_phaseB_f1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MobileNetV2
ax = axes[0]
epochs_mn = list(range(1, len(mn_s1_val_recall) + 1))
ax.plot(epochs_mn, mn_s1_val_recall, 'o-', color='steelblue', label='Val Recall (Malignant)')
ax.axvline(5.5, color='grey', linestyle='--', linewidth=0.8, label='Warm-up → Fine-tune')
ax.axhline(0.95, color='red', linestyle=':', linewidth=1, label='Target Recall (0.95)')
ax.set_title('MobileNetV2 — Stage 1 Val Recall', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Recall (Malignant)')
ax.set_ylim(0.9, 1.02); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ViT
ax = axes[1]
epochs_vit = list(range(1, len(vit_s1_val_f1) + 1))
ax.plot(epochs_vit, vit_s1_val_f1, 's-', color='tomato', label='Val Weighted F1')
ax.axvline(5.5, color='grey', linestyle='--', linewidth=0.8, label='Phase A → Phase B')
best_ep = vit_s1_val_f1.index(max(vit_s1_val_f1)) + 1
ax.axvline(best_ep, color='green', linestyle=':', linewidth=1,
           label=f'Best (ep {best_ep}: F1={max(vit_s1_val_f1):.4f})')
ax.set_title('ViT-B/16 — Stage 1 Val Weighted F1', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Weighted F1')
ax.set_ylim(0.7, 0.92); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Stage 1 Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage1_training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Stage 1 training curves saved.')

In [ ]:
# ── ViT-B/16 Stage 2 training (MobileNetV2 S2 curves not logged epoch-by-epoch)
vit_s2_phaseC_f1 = [0.5062, 0.7566, 0.7809, 0.8058, 0.8115]
vit_s2_phaseD_f1 = [0.8266, 0.8411, 0.8564, 0.8517, 0.8454,
                    0.8890, 0.8848, 0.8835, 0.8864, 0.8923]
vit_s2_val_f1 = vit_s2_phaseC_f1 + vit_s2_phaseD_f1

fig, ax = plt.subplots(figsize=(9, 4))
epochs_vit_s2 = list(range(1, len(vit_s2_val_f1) + 1))
ax.plot(epochs_vit_s2, vit_s2_val_f1, 's-', color='darkorange', label='Val Weighted F1')
ax.axvline(5.5, color='grey', linestyle='--', linewidth=0.8, label='Phase C → Phase D')
best_ep2 = vit_s2_val_f1.index(max(vit_s2_val_f1)) + 1
ax.axvline(best_ep2, color='green', linestyle=':', linewidth=1,
           label=f'Best (ep {best_ep2}: F1={max(vit_s2_val_f1):.4f})')
ax.set_title('ViT-B/16 — Stage 2 Val Weighted F1', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Weighted F1')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage2_training_vit.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Qualitative Evaluation — Grad-CAM vs Attention Rollout

**MobileNetV2** uses Grad-CAM on the last convolutional block to highlight
which image regions drove its decision. **ViT-B/16** uses Attention Rollout,
composing all 12 transformer block attention maps to find which 16×16 patches
the CLS token attended to.

The figure below shows three malignant test images (all melanoma).
MobileNetV2 focuses tightly on the lesion centre, while ViT distributes
attention more broadly but still centres on the lesion.

In [ ]:
comparison_fig = FIG_DIR / 'vit_vs_mobilenet_explainability.png'
if comparison_fig.exists():
    img = Image.open(comparison_fig)
    plt.figure(figsize=(14, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Loaded: {comparison_fig}')
else:
    print(f'Figure not found at {comparison_fig}')
    print('Run Section 11 of vit_pretrained_ham10000.ipynb to generate it.')

---
## 9. Efficiency Comparison

Measured on the same hardware using CPU inference (averaged over 100 forward passes)
to simulate mobile/edge deployment scenarios.

In [ ]:
efficiency = pd.DataFrame({
    'MobileNetV2': {
        'Parameters':          '2,226,434',
        'Stage 1 Size (MB)':   '8.73',
        'Stage 2 Size (MB)':   '8.73',
        'Inference (ms/img)':  '4.76',
    },
    'ViT-B/16 (pretrained)': {
        'Parameters':          '86,567,656',
        'Stage 1 Size (MB)':   '327.39',
        'Stage 2 Size (MB)':   '327.39',
        'Inference (ms/img)':  '124.59',
    },
})
efficiency.index.name = 'Metric'
print('\n' + '=' * 55)
print('  EFFICIENCY COMPARISON')
print('=' * 55)
print(efficiency.to_string())

# Bar chart — inference time comparison
fig, ax = plt.subplots(figsize=(6, 4))
models = ['MobileNetV2', 'ViT-B/16']
times  = [4.76, 124.59]
bars   = ax.bar(models, times, color=['steelblue', 'tomato'], width=0.5)
ax.set_ylabel('Inference time (ms / image)')
ax.set_title('CPU Inference Time per Image', fontsize=12, fontweight='bold')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{t} ms', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 145)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'inference_time_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nViT-B/16 is {:.0f}x slower per image than MobileNetV2.'.format(124.59/4.76))

---
## 10. Limitations

| Limitation | Impact |
|---|---|
| **Single dataset** | Results may not generalise beyond HAM10000. Clinical validation on independent datasets is needed. |
| **Class imbalance** | `nv` accounts for ~67% of images. Despite augmentation and class-weighted loss, rare classes (vasc, df) have fewer training examples. |
| **Stage 1 leakage** | Any malignant case missed by Stage 1 is never seen by Stage 2. MobileNetV2 avoids leakage by predicting nearly everything as malignant (recall=1.0), at the cost of 94% false positive rate. ViT balances precision and recall better. |
| **No clinical trial** | Neither model has been tested in a real clinical environment. Hair, rulers, and lighting artefacts may degrade performance. |
| **Hardware-specific timing** | Inference times were measured on one machine; real-world mobile performance will vary. |
| **Stage 2 ground truth for FP** | When Stage 1 incorrectly flags a benign image, Stage 2 receives an image with no meaningful subtype ground truth. These are clamped to class 0 (akiec) in pipeline evaluation. |

---
## 11. Summary

This project built a two-stage hierarchical pipeline for skin cancer detection on HAM10000.
Stage 1 classifies lesions as benign or malignant; Stage 2 subclassifies malignant lesions
into melanoma (MEL), basal cell carcinoma (BCC), or actinic keratosis (AKIEC).

**Key findings:**

- **MobileNetV2** achieves Stage 1 malignant recall = **1.00** — it catches every malignant
  case — but does so by flagging 94% of all images as malignant (1,412/1,503), giving low
  precision and poor overall accuracy. Stage 2 conditional accuracy is **0.63**.

- **ViT-B/16** achieves a more balanced Stage 1 result: malignant recall ~**0.67**, precision
  ~**0.60**, and a much higher overall accuracy (**0.87**). Stage 2 conditional accuracy is **0.90**.
  Its AUC of **0.90** confirms strong discriminative ability.

- **Efficiency:** MobileNetV2 is **26× faster** (4.76 ms vs 124.59 ms per image) and **37×
  smaller** (8.73 MB vs 327 MB). For mobile or point-of-care deployment, MobileNetV2 is the
  practical choice if its recall bias is acceptable; ViT is preferable for offline or
  high-throughput clinical screening where precision also matters.

**Future work:** Threshold tuning on MobileNetV2 Stage 1 to reduce false positives while
maintaining high recall; larger multi-institution datasets; model compression to bring ViT
closer to mobile deployment requirements.